In [ ]:
#Data
"""
movies
movieId,artistId,genre,date,awards
101,201,action,2024-06-25,5
102,202,comedy,2024-08-25,6
103,203,action,2024-06-25,2
103,204,horror,2024-06-25,5
104,205,horror,2024-08-25,1
105,206,action,2024-10-25,8
106,201,comedy,2024-06-25,9
107,202,action,2024-08-25,4
108,203,thriller,2024-10-25,5
109,204,action,2024-08-25,3
110,205,action,2024-06-25,7
111,206,action,2024-06-25,2
112,203,comedy,2024-10-25,5
113,204,thriller,2024-08-25,4
114,205,comedy,2024-06-25,4


artists
artistId,artistName,age,origin
201,Johnny,25,USA
202,Hugh,45,AU
203,Christian,45,UK
204,Will,48,USA
205,Scarlett,40,USA
206,Emilia,38,UK
207,Chris,34,AU
"""

In [1]:
#Import Libraries
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

In [2]:
#Create SparkSession with name "Movie Data Analysis"
spark = SparkSession.builder.appName("Movie Data Analysis").getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/07/03 06:12:53 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [9]:
#Create Dataframes (moviesdf & artistdf)
moviesdf=spark.read.csv("movies.csv",header=True,inferSchema=True)
artistdf=spark.read.csv("artists.csv",header=True,inferSchema=True)

moviesdf.show()

+-------+--------+--------+----------+------+
|movieId|artistId|   genre|      date|awards|
+-------+--------+--------+----------+------+
|    101|     201|  action|2024-06-25|     5|
|    102|     202|  comedy|2024-08-25|     6|
|    103|     203|  action|2024-06-25|     2|
|    103|     204|  horror|2024-06-25|     5|
|    104|     205|  horror|2024-08-25|     1|
|    105|     206|  action|2024-10-25|     8|
|    106|     201|  comedy|2024-06-25|     9|
|    107|     202|  action|2024-08-25|     4|
|    108|     203|thriller|2024-10-25|     5|
|    109|     204|  action|2024-08-25|     3|
|    110|     205|  action|2024-06-25|     7|
|    111|     206|  action|2024-06-25|     2|
|    112|     203|  comedy|2024-10-25|     5|
|    113|     204|thriller|2024-08-25|     4|
|    114|     205|  comedy|2024-06-25|     4|
+-------+--------+--------+----------+------+



In [11]:
#Filter moviesdf for genre = horror
moviesdf.createOrReplaceTempView("movies_table")
df_sql=spark.sql("""
    select * from movies_table
    where genre=="horror"
""")
df_sql.show()

+-------+--------+------+----------+------+
|movieId|artistId| genre|      date|awards|
+-------+--------+------+----------+------+
|    103|     204|horror|2024-06-25|     5|
|    104|     205|horror|2024-08-25|     1|
+-------+--------+------+----------+------+



In [20]:
#Count movies per artist
artistdf.createOrReplaceTempView("artists_table")
df_sql1=spark.sql("""
    select a.artistId,artistName, count(movieId) as no_of_movies 
    from movies_table m
    inner join artists_table a
    on m.artistId=a.artistId

    group by a.artistId,artistName
""")
df_sql1.show()

+--------+----------+------------+
|artistId|artistName|no_of_movies|
+--------+----------+------------+
|     206|    Emilia|           2|
|     201|    Johnny|           2|
|     202|      Hugh|           2|
|     203| Christian|           3|
|     205|  Scarlett|           3|
|     204|      Will|           3|
+--------+----------+------------+



In [21]:
#Count total movies for action genre
df_sql2=spark.sql("""
    select count(movieId) from movies_table
    where genre=="action"
""")
df_sql2.show()



+--------------+
|count(movieId)|
+--------------+
|             7|
+--------------+



In [22]:
#Join moviesdf & artistdf using inner join on artistId
df1 = moviesdf.join(artistdf,on="artistId",how="inner")
df1.show()

+--------+-------+--------+----------+------+----------+---+------+
|artistId|movieId|   genre|      date|awards|artistName|age|origin|
+--------+-------+--------+----------+------+----------+---+------+
|     201|    101|  action|2024-06-25|     5|    Johnny| 25|   USA|
|     202|    102|  comedy|2024-08-25|     6|      Hugh| 45|    AU|
|     203|    103|  action|2024-06-25|     2| Christian| 45|    UK|
|     204|    103|  horror|2024-06-25|     5|      Will| 48|   USA|
|     205|    104|  horror|2024-08-25|     1|  Scarlett| 40|   USA|
|     206|    105|  action|2024-10-25|     8|    Emilia| 38|    UK|
|     201|    106|  comedy|2024-06-25|     9|    Johnny| 25|   USA|
|     202|    107|  action|2024-08-25|     4|      Hugh| 45|    AU|
|     203|    108|thriller|2024-10-25|     5| Christian| 45|    UK|
|     204|    109|  action|2024-08-25|     3|      Will| 48|   USA|
|     205|    110|  action|2024-06-25|     7|  Scarlett| 40|   USA|
|     206|    111|  action|2024-06-25|     2|   

In [31]:
#Fetch the most popular genre (output should be single string value like - horror)

df2=moviesdf.orderBy(F.desc("awards")).select("genre").collect()[0][0]
df2

'comedy'

In [33]:
#Filter artistdf for Origin - DN
artistdf.filter(F.col("origin")=="DN").show()

+--------+----------+---+------+
|artistId|artistName|age|origin|
+--------+----------+---+------+
+--------+----------+---+------+



In [40]:
#Find total awards given (output should be a number)
moviesdf.agg(F.sum("awards")).collect()[0][0]

70

In [45]:
#Find three latest released movie (output should be dataframe)
moviesdf.orderBy(F.desc("date")).limit(3).show()

+-------+--------+--------+----------+------+
|movieId|artistId|   genre|      date|awards|
+-------+--------+--------+----------+------+
|    105|     206|  action|2024-10-25|     8|
|    108|     203|thriller|2024-10-25|     5|
|    112|     203|  comedy|2024-10-25|     5|
+-------+--------+--------+----------+------+



In [47]:
#Find the most awarded movie (Output should be movieId - integer)
moviesdf.orderBy(F.desc("awards")).first()[0]

106

In [49]:
#Find the top two genres (Output should be list)
moviesdf.orderBy(F.desc("awards")).limit(2).show()

+-------+--------+------+----------+------+
|movieId|artistId| genre|      date|awards|
+-------+--------+------+----------+------+
|    106|     201|comedy|2024-06-25|     9|
|    105|     206|action|2024-10-25|     8|
+-------+--------+------+----------+------+

